In [6]:
import pandas as pd, numpy as np
import pulp

In [ ]:
df_production = pd.read_excel(r"C:\Users\LENOVO\Downloads\production_planning_dataset.xlsx")
products = pd.read_excel(r"C:\Users\LENOVO\Downloads\production_planning_dataset.xlsx",sheet_name="Products")
demand = pd.read_excel(r"C:\Users\LENOVO\Downloads\production_planning_dataset.xlsx",sheet_name="Demand")
capacity = pd.read_excel(r"C:\Users\LENOVO\Downloads\production_planning_dataset.xlsx",sheet_name="Capacity")
initial_inventory = pd.read_excel(r"C:\Users\LENOVO\Downloads\production_planning_dataset.xlsx",sheet_name="Initial Inventory")
SKUs = products["SKU"].tolist()
month = demand["Month"].tolist()
demand_dict = {}

production_cost = dict(zip(products["SKU"], products["Production_Cost_per_unit"]))
production_hour = dict(zip(products["SKU"], products["Production_Time_hr_per_unit"]))
holding_cost = dict(zip(products["SKU"], products["Holding_Cost_per_unit_month"]))
max_inventory = dict(zip(products["SKU"], products["Max_Inventory"]))
capacity_dict = dict(zip(capacity["Month"], capacity["Available_Production_Hours"]))
initial_inventory_dict = dict(zip(initial_inventory["SKU"], initial_inventory["Initial_Inventory"]))

for _, row in demand.iterrows():
    for sku in SKUs:
        demand_dict[sku, row["Month"]] = row[sku]

In [12]:
def optimize_production(demand_change=0, capacity_change=0, holding_cost_change=0):

    production_result = []
    inventory_result = []

    demand_scenario_dict = {}
    capacity_scenario_dict = {}
    holding_cost_scenario_dict ={}

    for _, row in demand.iterrows():
        for sku in SKUs:
            demand_scenario_dict[sku, row["Month"]] = (
                row[sku] * (1 + demand_change)
            )
    for t in month:
        capacity_scenario_dict[t] = capacity_dict[t] * (1 + capacity_change)

    for sku in SKUs: 
        holding_cost_scenario_dict[sku] = holding_cost[sku] * (1 + holding_cost_change)

    model = pulp.LpProblem("Production Planning", pulp.LpMinimize)

    #Khai báo Decision Variables 
    production = pulp.LpVariable.dicts("Production", [(p,t) for p in SKUs for t in month], lowBound=0)
    inventory = pulp.LpVariable.dicts("Inventory", [(p,t) for p in SKUs for t in month], lowBound=0)

    #Objective function
    model += pulp.lpSum(production_cost[p] * production[p,t] + holding_cost_scenario_dict[p] * inventory[p,t] for p in SKUs for t in month)

    #Constraints
    #Inventory balance constraint

    for p in SKUs:
        for i, t in enumerate(month):
            if i == 0:
                beginning_inventory = initial_inventory_dict[p]
            else:
                prev_month = month[i - 1]
                beginning_inventory = inventory[p, prev_month]

            model += (beginning_inventory + production[p,t] - demand_scenario_dict[p,t] == inventory[p,t])

    #Capacity constraint

    for t in month: 
        model += pulp.lpSum(production[p,t] * production_hour[p] for p in SKUs) <= capacity_scenario_dict[t]

    #Maximum inventory constraint

    for p in SKUs: 
        for t in month: 
            model += inventory[p, t] <= max_inventory[p]

    model.solve()
    print("Status:", pulp.LpStatus[model.status])

    if pulp.LpStatus[model.status] != "Optimal":
        return None, None

    for p in SKUs:
        for t in month: 
            production_result.append({
                "SKU":p,
                "Month":t,
                "Production": production[p,t].value()
            })
            inventory_result.append({
                "SKU":p,
                "Month": t,
                "Ending Inventory": inventory[p,t].value()
            })

    production_df = pd.DataFrame(production_result)
    inventory_df = pd.DataFrame(inventory_result)

    production_df["Production Cost"] = production_df.apply(lambda row: row["Production"] * production_cost[row["SKU"]], axis=1)
    total_production_cost = production_df["Production Cost"].sum()

    inventory_df["Holding Cost"] = inventory_df.apply(lambda row: row["Ending Inventory"] * holding_cost_scenario_dict[row["SKU"]], axis=1)
    total_inventory_cost = inventory_df["Holding Cost"].sum()

    total_cost = total_inventory_cost + total_production_cost

    average_inventory = inventory_df["Ending Inventory"].mean()

    production_df["Production Hours"] = production_df.apply(lambda row: row["Production"] * production_hour[row["SKU"]], axis=1)
    monthly_production_hours = (production_df.groupby("Month")["Production Hours"].sum())
    capacity_utilization = {t: monthly_production_hours[t]/ capacity_scenario_dict[t] for t in month}
    average_capacity_utilization = sum(capacity_utilization.values())/len(capacity_utilization)
    average_capacity_utilization_pct = (average_capacity_utilization*100)

    summary = {
        "Total Cost": total_cost,
        "Production Cost": total_production_cost,
        "Holding Cost": total_inventory_cost,
        "Average Inventory": average_inventory,
        "Average Capacity Utilization (%)": average_capacity_utilization_pct
    }
    return production_df, inventory_df, summary


In [21]:
base_production, base_inventory, base_summary = optimize_production()

demand20_production, demand20_inventory, demand20_summary = optimize_production(
    demand_change=0.20
)

capacity20_production, capacity20_inventory, capacity20_summary = optimize_production(
    capacity_change=-0.20
)

holding50_production, holding50_inventory, holding50_summary = optimize_production(
    holding_cost_change=0.50
)

scenario_results = pd.DataFrame([
    {
        "Scenario": "Base",
        **base_summary
    },
    {
        "Scenario": "Demand +20%",
        **demand20_summary
    },
    {
        "Scenario": "Capacity -20%",
        **capacity20_summary
    },
    {
        "Scenario": "Holding Cost +50%",
        **holding50_summary
    }
])

base_cost = base_summary["Total Cost"]

scenario_results["Cost Change %"] = (
    (scenario_results["Total Cost"] - base_cost)
    / base_cost
    * 100
)

scenario_results.style.format({
    "Total Cost": "${:,.0f}",
    "Production Cost": "${:,.0f}",
    "Holding Cost": "${:,.0f}",
    "Average Inventory": "{:,.0f}",
    "Average Capacity Utilization (%)": "{:.1f}%",
    "Cost Change %": "{:+.1f}%"
})

c:\Users\LENOVO\anaconda3\Lib\site-packages\pulp\pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")
c:\Users\LENOVO\anaconda3\Lib\site-packages\pulp\pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")
c:\Users\LENOVO\anaconda3\Lib\site-packages\pulp\pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


Status: Optimal
Status: Optimal
Status: Optimal
Status: Optimal


c:\Users\LENOVO\anaconda3\Lib\site-packages\pulp\pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


,Scenario,Total Cost,Production Cost,Holding Cost,Average Inventory,Average Capacity Utilization (%),Cost Change %
0,Base,"$320,516","$320,414",$102,4,75.9%,+0.0%
1,Demand +20%,"$389,476","$385,364","$4,113",175,90.6%,+21.5%
2,Capacity -20%,"$326,667","$320,414","$6,253",276,94.0%,+1.9%
3,Holding Cost +50%,"$320,568","$320,414",$154,4,75.9%,+0.0%
